<a href="https://colab.research.google.com/github/HashiniDivyanjanee/Hashini-Divyanjanee-HPC-Task-2638122/blob/main/Task_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile matrix.c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <limits.h>
#include <omp.h>

typedef struct {
    int rows, cols;
    double *data;           // rows * cols
} Matrix;

#define AT(m, i, j) ((m)->data[(size_t)(i) * (m)->cols + (j)])

// memory helpers
Matrix *allocMatrix(int rows, int cols) {
    Matrix *m = malloc(sizeof(Matrix));
    if (!m) return NULL;
    m->rows = rows;
    m->cols = cols;
    m->data = malloc((size_t)rows * cols * sizeof(double));
    if (!m->data) { free(m); return NULL; }
    return m;
}

void freeMatrix(Matrix *m) {
    if (m) { free(m->data); free(m); }
}

// reading
// reads next char, skipping spaces/tabs/carriage returns; returns EOF at end of file
static int nextChar(FILE *fp) {
    char c;
    int s;
    do { s = fscanf(fp, "%c", &c); } while (s == 1 && (c == ' ' || c == '\t' || c == '\r'));
    return (s == 1) ? (unsigned char)c : EOF;
}

// returns 1 = matrix read, 0 = end of file, -1 = error (message already printed)
int readMatrix(FILE *fp, int index, Matrix **out) {
    int r, c;
    int n = fscanf(fp, " %d,%d", &r, &c);
    if (n == EOF) return 0;
    if (n != 2) {
        fprintf(stderr, "Error: invalid header for matrix %d (expected rows,cols)\n", index);
        return -1;
    }
    if (r <= 0 || c <= 0) {
        fprintf(stderr, "Error: matrix %d has invalid size %d,%d\n", index, r, c);
        return -1;
    }
    Matrix *m = allocMatrix(r, c);
    if (!m) {
        fprintf(stderr, "Error: out of memory for matrix %d\n", index);
        return -1;
    }
    for (int i = 0; i < r; i++) {
        for (int j = 0; j < c; j++) {
            double v;
            if (fscanf(fp, " %lf", &v) != 1) {
                fprintf(stderr, "Error: matrix %d, row %d, column %d: missing or non-numeric value\n",
                        index, i + 1, j + 1);
                freeMatrix(m);
                return -1;
            }
            AT(m, i, j) = v;

            int ch = nextChar(fp);
            if (j < c - 1) {
                if (ch != ',') {
                    fprintf(stderr, "Error: matrix %d, row %d has fewer than %d values\n", index, i + 1, c);
                    freeMatrix(m);
                    return -1;
                }
            } else if (ch != '\n' && ch != EOF) {
                fprintf(stderr, "Error: matrix %d, row %d has more than %d values\n", index, i + 1, c);
                freeMatrix(m);
                return -1;
            }
        }
    }
    *out = m;
    return 1;
}

// writing
void writeMatrix(FILE *out, const char *name, const Matrix *m) {
    fprintf(out, "%s - %d,%d\n", name, m->rows, m->cols);
    for (int i = 0; i < m->rows; i++) {
        for (int j = 0; j < m->cols; j++) {
            double v = AT(m, i, j);
            if (isnan(v)) fprintf(out, "NaN");
            else          fprintf(out, "%.6f", v);
            if (j < m->cols - 1) fprintf(out, ",");
        }
        fprintf(out, "\n");
    }
    fprintf(out, "\n");
}

// operations (OpenMP)
// never use more threads than there are rows to share out
static int capThreads(int requested, int rows) {
    return requested < rows ? requested : rows;
}

Matrix *elementwise(const Matrix *A, const Matrix *B, char op, int requested, int *used) {
    Matrix *R = allocMatrix(A->rows, A->cols);
    if (!R) return NULL;
    int t = capThreads(requested, R->rows);
    *used = t;
    #pragma omp parallel for num_threads(t) schedule(static)
    for (int i = 0; i < R->rows; i++) {
        for (int j = 0; j < R->cols; j++) {
            double a = AT(A, i, j), b = AT(B, i, j), v = 0.0;
            switch (op) {
                case '+': v = a + b; break;
                case '-': v = a - b; break;
                case '*': v = a * b; break;
                case '/': v = (b == 0.0) ? NAN : a / b; break;
            }
            AT(R, i, j) = v;
        }
    }
    return R;
}

Matrix *transpose(const Matrix *A, int requested, int *used) {
    Matrix *R = allocMatrix(A->cols, A->rows);
    if (!R) return NULL;
    int t = capThreads(requested, R->rows);
    *used = t;
    #pragma omp parallel for num_threads(t) schedule(static)
    for (int i = 0; i < R->rows; i++)
        for (int j = 0; j < R->cols; j++)
            AT(R, i, j) = AT(A, j, i);
    return R;
}

Matrix *multiply(const Matrix *A, const Matrix *B, int requested, int *used) {
    Matrix *R = allocMatrix(A->rows, B->cols);
    if (!R) return NULL;
    int t = capThreads(requested, R->rows);
    *used = t;
    #pragma omp parallel for num_threads(t) schedule(static)
    for (int i = 0; i < R->rows; i++) {
        for (int j = 0; j < R->cols; j++) {
            double sum = 0.0;
            for (int k = 0; k < A->cols; k++)
                sum += AT(A, i, k) * AT(B, k, j);
            AT(R, i, j) = sum;
        }
    }
    return R;
}

// run one operation, write it to the file, free the result
static int runOp(FILE *out, const char *name, Matrix *R, const int *used) {
    if (!R) {
        fprintf(stderr, "Error: out of memory during '%s'\n", name);
        return -1;
    }
    printf("  %-32s threads used: %d\n", name, *used);
    writeMatrix(out, name, R);
    freeMatrix(R);
    return 0;
}

int processPair(FILE *out, const Matrix *A, const Matrix *B, int pair, int requested) {
    int used = 0;
    fprintf(out, "...................... Pair %d: A (%d,%d) and B (%d,%d) ......................\n\n",
            pair, A->rows, A->cols, B->rows, B->cols);
    printf("Pair %d: A (%d,%d), B (%d,%d)\n", pair, A->rows, A->cols, B->rows, B->cols);

    int sameShape = (A->rows == B->rows && A->cols == B->cols);

    if (sameShape) {
        if (runOp(out, "Addition",                    elementwise(A, B, '+', requested, &used), &used)) return -1;
        if (runOp(out, "Subtraction",                 elementwise(A, B, '-', requested, &used), &used)) return -1;
        if (runOp(out, "Element-wise multiplication", elementwise(A, B, '*', requested, &used), &used)) return -1;
        if (runOp(out, "Element-wise division",       elementwise(A, B, '/', requested, &used), &used)) return -1;
    } else {
        fprintf(out, "Addition cannot be done (shapes differ).\n\n");
        fprintf(out, "Subtraction cannot be done (shapes differ).\n\n");
        fprintf(out, "Element-wise multiplication cannot be done (shapes differ).\n\n");
        fprintf(out, "Element-wise division cannot be done (shapes differ).\n\n");
    }

    if (runOp(out, "Transpose of A", transpose(A, requested, &used), &used)) return -1;
    if (runOp(out, "Transpose of B", transpose(B, requested, &used), &used)) return -1;

    if (A->cols == B->rows) {
        if (runOp(out, "Matrix multiplication (A x B)", multiply(A, B, requested, &used), &used)) return -1;
    } else {
        fprintf(out, "Matrix multiplication (A x B) cannot be done (A columns != B rows).\n\n");
    }
    return 0;
}

// main
int main(int argc, char *argv[]) {
    if (argc != 3) {
        printf("Usage: %s <input_file> <number_of_threads>\n", argv[0]);
        return 1;
    }

    char *end;
    long requested = strtol(argv[2], &end, 10);
    if (*end != '\0' || requested <= 0 || requested > INT_MAX) {
        fprintf(stderr, "Error: number of threads must be a positive integer\n");
        return 1;
    }

    FILE *fp = fopen(argv[1], "r");
    if (!fp) {
        fprintf(stderr, "Error: cannot open input file '%s'\n", argv[1]);
        return 1;
    }
    FILE *out = fopen("results.txt", "w");
    if (!out) {
        fprintf(stderr, "Error: cannot create results.txt\n");
        fclose(fp);
        return 1;
    }

    int pair = 0, status = 0;
    while (1) {
        Matrix *A = NULL, *B = NULL;

        int rc = readMatrix(fp, 2 * pair + 1, &A);
        if (rc == 0) break;
        if (rc < 0) { status = 1; break; }

        rc = readMatrix(fp, 2 * pair + 2, &B);
        if (rc == 0) {
            fprintf(out, "Matrix %d has no partner matrix, so it was skipped.\n", 2 * pair + 1);
            printf("Matrix %d has no partner matrix, skipped.\n", 2 * pair + 1);
            freeMatrix(A);
            break;
        }
        if (rc < 0) { freeMatrix(A); status = 1; break; }

        pair++;
        if (processPair(out, A, B, pair, (int)requested) != 0) status = 1;
        freeMatrix(A);
        freeMatrix(B);
        if (status) break;
    }

    fclose(fp);
    fclose(out);
    if (status == 0) printf("Done. %d pair(s) written to results.txt\n", pair);
    else             printf("Stopped because of an error. Results so far are in results.txt\n");
    return status;
}

Writing matrix.c


In [2]:
!gcc -fopenmp -O2 -o matrix matrix.c -lm

In [3]:
!./matrix MatData.txt 4

Pair 1: A (3,4), B (3,4)
  Addition                         threads used: 3
  Subtraction                      threads used: 3
  Element-wise multiplication      threads used: 3
  Element-wise division            threads used: 3
  Transpose of A                   threads used: 4
  Transpose of B                   threads used: 4
Pair 2: A (4,6), B (6,2)
  Transpose of A                   threads used: 4
  Transpose of B                   threads used: 2
  Matrix multiplication (A x B)    threads used: 4
Pair 3: A (5,5), B (5,5)
  Addition                         threads used: 4
  Subtraction                      threads used: 4
  Element-wise multiplication      threads used: 4
  Element-wise division            threads used: 4
  Transpose of A                   threads used: 4
  Transpose of B                   threads used: 4
  Matrix multiplication (A x B)    threads used: 4
Pair 4: A (8,4), B (4,8)
  Transpose of A                   threads used: 4
  Transpose of B                 

In [4]:
!head -40 results.txt

...................... Pair 1: A (3,4) and B (3,4) ......................

Addition - 3,4
903710.476323,1008456.397449,1301741.130630,344575.269830
980489.233941,642184.151049,1218029.777154,1319312.798389
882116.994696,1431965.282759,1433481.668147,776431.887252

Subtraction - 3,4
-711535.555473,378723.889685,-557316.945936,-38856.812490
262209.847295,6933.747073,-449269.261636,670789.294261
-353473.282246,243725.088597,262291.865645,-676854.689106

Element-wise multiplication - 3,4
77602444578.418213,218388130234.419861,345981948236.610107,29305566175.383572
223151283463.935425,103088101752.513138,320438417145.716553,322656995674.460938
163296757767.435638,497780663053.861816,496518167532.492798,36178551344.230629

Element-wise division - 3,4
0.118976,2.202809,0.400431,0.797321
1.730105,1.021830,0.461081,3.068666
0.427847,1.410229,1.447907,0.068519

Transpose of A - 4,3
96087.460425,621349.540618,264321.856225
693590.143567,324558.949061,837845.185678
372212.092347,384380.257759,8478